# Association Rule Mining (TOR) Pipeline — Social Media (Tweets)

This notebook executes the end-to-end NLP and semantic association pipeline for social media data:
1. **Specialized Social Preprocessing**: cleaning URLs, mentions (@), hashtags (#), emojis, and expanding informal abbreviations.
2. **Trie of Rules (TOR) Construction**: frequent itemset mining and direct **vector PDF** graph visualization.
3. **Cluster Analysis**: core/peripheral partitioning and hierarchical branch extraction.
4. **Content Mapping**: associating original tweets with each rule cluster and exporting individual CSVs.
5. **AI Synthesis**: generating structured prompts for Large Language Models (LLMs).
6. **Auxiliary Visualizations**: word frequency rankings, frequency bar plots, and co-occurrence graphs (`.gdf`).

In [ ]:
import os
import pandas as pd
import networkx as nx
import utils.pre_processing_social_media as psm
import utils.analize_tor as ator
import utils.processing_file as pf
from utils.pipeline import (
    garantir_diretorios,
    gerar_tor,
    separar_e_salvar_clusters,
    salvar_visualizacoes_comuns
)

# Initialize standard output directories
garantir_diretorios()

## 1. Parameters & Configuration

In [ ]:
# Dataset base filename in process_datasets/ (without .csv extension)
FILE_NAME = "vacinal_2025_05"

# Association Rule Mining Algorithm: 'Apriori', 'FP-max', or 'FP-growth'
ALG = "Apriori"
MIN_SUPPORT = 0.009

# Target cluster index for AI prompt generation
CLUSTER_IA = 10

INPUT_FILE = os.path.join("process_datasets", f"{FILE_NAME}.csv")
OUTPUT_IDENTIFIER = f"{FILE_NAME}_{ALG}_{MIN_SUPPORT}"
OUTPUT_GRAPHML = os.path.join("outputs", f"{OUTPUT_IDENTIFIER}.graphml")
OUTPUT_PDF = os.path.join("outputs", f"{OUTPUT_IDENTIFIER}_grafo.pdf")

## 2. Preprocessing & Trie of Rules (TOR) Generation

In [ ]:
print(f"Preprocessing database: {INPUT_FILE}...")
pre_processing_result, txt_database = psm.pre_processing_database(INPUT_FILE)
print(f"Total tweets processed: {len(pre_processing_result)}")

# Build TOR and save .graphml file
tor = gerar_tor(pre_processing_result, alg=ALG, min_support=MIN_SUPPORT, output_graphml=OUTPUT_GRAPHML)

# Render and export ForceAtlas2 graph to PDF and display inline
G = nx.read_graphml(OUTPUT_GRAPHML)
pf.plotar_grafo(G, output_pdf=OUTPUT_PDF, show=True)

## 3. Hierarchical Cluster Analysis & Tweet Mapping

In [ ]:
# 1. Separate central core from peripheral clusters
G_central, G_perifericos, path_central, path_perifericos = separar_e_salvar_clusters(
    graphml_path=OUTPUT_GRAPHML,
    file_name=OUTPUT_IDENTIFIER
)

# 2. Extract clusters with conceptual branches
print("\nExtracting conceptual branches from clusters...")
all_clusters = ator.extrair_clusters(OUTPUT_GRAPHML, show_results=True)

# 3. Map original tweets containing rules of each cluster
print("\nMapping tweets to corresponding clusters...")
tweets_clusters = ator.tweets_clusters(
    clusters=all_clusters,
    textos_tokenizados=pre_processing_result,
    textos_originais=txt_database
)

# 4. Save cluster-specific tweets into CSV files
pasta_csv = "outputs_clusters_csv"
os.makedirs(pasta_csv, exist_ok=True)
for c in tweets_clusters:
    if c["n_tweets"] == 0:
        continue
    cluster_name = c["raiz"].replace(" ", "_")[:30]
    df_cluster = pd.DataFrame(c["tweets"])
    csv_path = os.path.join(pasta_csv, f"{OUTPUT_IDENTIFIER}_cluster_{c['cluster_idx']}_{cluster_name}.csv")
    df_cluster.to_csv(csv_path, index=False)

print(f"\nIndividual cluster CSV files saved to: {pasta_csv}/")

## 4. Synthesis & AI (LLM) Prompt Generation

In [ ]:
if all_clusters and 1 <= CLUSTER_IA <= len(all_clusters):
    selected_cluster = all_clusters[CLUSTER_IA - 1]
    prompt = ator.prompt_IA(selected_cluster)
    print(f"--- Generated Prompt for Cluster {CLUSTER_IA} ({selected_cluster['raiz']}) ---\n")
    print(prompt)
else:
    print(f"Cluster {CLUSTER_IA} not found or cluster list is empty.")

## 5. Auxiliary Visualizations

In [ ]:
# Simplified preprocessing for word rankings and co-occurrence
pre_processing_result_simples, _ = psm.pre_processing_database(
    file_path=INPUT_FILE,
    simples=True
)

# Export word ranking CSV, co-occurrence .gdf graph, and plot frequency distribution
df_ranking, file_ranking, file_grafo = salvar_visualizacoes_comuns(
    dados_simples=pre_processing_result_simples,
    file_name=OUTPUT_IDENTIFIER,
    pasta_ranking="ranking_palavras",
    plotar_frequencia=True
)

# Continuous text sample for Word Tree
print("\nContinuous text preview for Word Tree (first 500 characters):")
print(" ".join(txt_database)[:500] + "...")